# Round5 Exit Recall Specialist

목표: **exit Recall 최우선**. 같은 원본의 변형본이 train/val/test에 섞이지 않도록 source-group split을 다시 만들고, Round2에서 시작해 exit-positive human-GT를 집중 보강합니다. 최종 MVP는 Round5가 exit만 담당합니다.

In [ ]:
# Cell 1 - 패키지/Drive/설치
!pip -q install ultralytics==8.4.121 pyyaml opencv-python-headless
from google.colab import drive,files
drive.mount('/content/drive')
from pathlib import Path
import zipfile,sys,json
PKG=Path('/content/round5_exit_recall')
if not PKG.exists():
    zs=list(Path('/content').glob('round5_exit_recall*.zip'))
    if not zs:
        print('round5_exit_recall.zip을 선택하세요.')
        up=files.upload(); zs=[Path('/content')/x for x in up if x.endswith('.zip')]
    if not zs: raise RuntimeError('ZIP 없음')
    with zipfile.ZipFile(zs[0]) as z:z.extractall('/content')
assert (PKG/'models/round2_best.pt').exists()
sys.path.insert(0,str(PKG/'scripts'))
print('✅',PKG)

In [ ]:
# Cell 2 - 설정/데이터 확인
from common import load_cfg
CONFIG=PKG/'config.yaml';cfg=load_cfg(CONFIG)
from pathlib import Path
src=Path(cfg['paths']['source_dataset_root'])
for p in [src/'images/train',src/'labels/train',src/'images/val',src/'labels/val']:print('✅' if p.exists() else '❌',p)
if not (src/'images/train').exists():raise RuntimeError('config.yaml의 source_dataset_root를 확인하세요.')

In [ ]:
# Cell 3 - source-group locked split
from build_split import run
r=run(cfg);print(json.dumps(r,indent=2,ensure_ascii=False))

In [ ]:
# Cell 4 - exit 집중 증강
from augment import run
r=run(cfg);print(json.dumps(r,indent=2,ensure_ascii=False))

In [ ]:
# Cell 5 - Stage A/B 학습
import torch
print('CUDA',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
from train import run
print(run(cfg,PKG))

In [ ]:
# Cell 6 - exit Recall 중심 모델/threshold 선택 + locked test
from evaluate import run
res=run(cfg,PKG)
print('SELECTED:',res['selected'])
print('VAL:',res['selected_val_metrics'])
print('TEST:',res['test_metrics'])
print('FINAL:',res['final_pt'])

In [ ]:
# Cell 7 - 최종 산출물
OUT=Path(cfg['paths']['output_root']);print('Output:',OUT)
for x in ['exit_specialist_best.pt','exit_specialist_best.onnx','exit_threshold.json','selection_report.json','split_report.json','augmentation_report.json']:
    print('✅' if (OUT/x).exists() else '⚠️',x)
print('MVP v2: exit=Round5 full+tiled, stair/you_are_here=Round4, 나머지=Round2')

In [ ]:
import json
from pathlib import Path

report_path = Path(
    "/content/drive/MyDrive/evacuation_yolo/"
    "round5_exit_recall/selection_report.json"
)

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print(json.dumps(report, indent=2, ensure_ascii=False))